# Reconstruction Analysis (Load Trained Model)
Load a saved Linear AE or AE from disk/Drive, recreate the model from the results JSON, analyze latent distributions, and compute reconstruction metrics (MSE, MAE, Frobenius).

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Step 1: Load Train/Val/Test Splits
Load correlation matrices from an existing dataset folder containing `train.pt`, `val.pt`, and `test.pt`.
- Local PC: `data/processed/dataset/<DATASET_NAME>`
- Google Colab: `dataset_tesi/<DATASET_NAME>`

In [ ]:
DATASET_NAME = 'data_00_20_w724_s10'

if 'google.colab' in sys.modules:
    print('Environment: Google Colab')
    IS_COLAB = True
else:
    print('Environment: Local PC')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    base_dir = DRIVE_ROOT / 'dataset_tesi'
else:
    project_root = Path.cwd().resolve().parent
    DRIVE_ROOT = None
    base_dir = project_root / 'data' / 'processed' / 'dataset'

dataset_dir = base_dir / DATASET_NAME

TRAIN_FILE = dataset_dir / 'train.pt'
VAL_FILE = dataset_dir / 'val.pt'
TEST_FILE = dataset_dir / 'test.pt'

for split_path in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    if not split_path.exists():
        raise FileNotFoundError(
            f"File '{split_path.name}' not found in: {dataset_dir.absolute()}"
        )

print(f'Dataset selected: {DATASET_NAME}')
print(f'Train: {TRAIN_FILE.name} | Val: {VAL_FILE.name} | Test: {TEST_FILE.name}')

In [ ]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), meta


train_corr, train_meta = load_corr_payload(TRAIN_FILE)
val_corr, val_meta = load_corr_payload(VAL_FILE)
test_corr, test_meta = load_corr_payload(TEST_FILE)

if train_corr.shape[1:] != test_corr.shape[1:]:
    raise ValueError(f'Train/test asset dims mismatch: {train_corr.shape} vs {test_corr.shape}')
if val_corr.shape[1:] != train_corr.shape[1:]:
    raise ValueError(f'Val/train asset dims mismatch: {val_corr.shape} vs {train_corr.shape}')

print(f'train_corr shape: {tuple(train_corr.shape)}')
print(f'val_corr shape:   {tuple(val_corr.shape)}')
print(f'test_corr shape:  {tuple(test_corr.shape)}')

## Step 2: Prepare Matrices from Existing Splits
Use full correlation matrices as input. Each matrix is flattened to a vector of size `N x N`.
Train/val/test come directly from the loaded split files (no random split in notebook).

In [ ]:
train_np = train_corr.numpy().astype(np.float32)
val_np = val_corr.numpy().astype(np.float32)
test_np = test_corr.numpy().astype(np.float32)

n_train, n_assets, _ = train_np.shape
n_val = val_np.shape[0]
n_test = test_np.shape[0]
n_matrices = n_train + n_val + n_test
n_features = n_assets * n_assets

x_train = torch.from_numpy(train_np.reshape(n_train, n_features))
x_val = torch.from_numpy(val_np.reshape(n_val, n_features))
x_test = torch.from_numpy(test_np.reshape(n_test, n_features))

VAL_FRACTION = n_val / n_matrices
TEST_FRACTION = n_test / n_matrices

print(f'Number of matrices: {n_matrices}')
print(f'Matrix shape: ({n_assets}, {n_assets})')
print(f'Flattened input size: {n_features}')
print(f'Train size: {n_train} | Val size: {n_val} | Test size: {n_test}')

## Step 3: Define the Models
Use a Linear Autoencoder (no activations) or a non-linear AutoEncoder (with hidden layers).

In [ ]:
class LinearAutoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Linear(input_dim, latent_dim)
        self.decoder = nn.Linear(latent_dim, input_dim)

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]

        dimensions = [input_dim, *hidden_dims, latent_dim]

        encoder_layers = []
        for i in range(len(dimensions) - 1):
            encoder_layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
            if i < len(dimensions) - 2:
                encoder_layers.append(nn.ReLU())
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_dims = dimensions[::-1]
        decoder_layers = []
        for i in range(len(decoder_dims) - 1):
            decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
            if i < len(decoder_dims) - 2:
                decoder_layers.append(nn.ReLU())
            else:
                decoder_layers.append(nn.Tanh())
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

## Step 4: Load Best Model from Results JSON
Select the results JSON path, load model details, recreate the architecture, and load weights.

In [ ]:
RESULTS_JSON_PATH = 'best_models_tesi/AE/AE_07/AE_results_AE_07.json'
MODEL_TYPE = None  # Set to 'linear' or 'ae' to override detection
WEIGHTS_OVERRIDE_PATH = None  # Optional: set a .pt path if JSON does not include it


def resolve_path(path_str, base_dir=None, results_dir=None):
    if path_str is None:
        return None
    p = Path(path_str)
    if p.is_absolute() and p.exists():
        return p.resolve()

    if results_dir is not None:
        candidate = results_dir / p
        if candidate.exists():
            return candidate.resolve()

    if base_dir is not None:
        candidate = base_dir / p
        if candidate.exists():
            return candidate.resolve()

    normalized = str(path_str).replace('\\', '/')
    if base_dir is not None and 'best_models_tesi' in normalized:
        rel = normalized[normalized.index('best_models_tesi'):]
        candidate = base_dir / rel
        if candidate.exists():
            return candidate.resolve()

    return p


results_path = Path(RESULTS_JSON_PATH)
if not results_path.is_absolute():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        results_path = (DRIVE_ROOT / results_path).resolve()
    else:
        results_path = (project_root / results_path).resolve()

if not results_path.exists():
    raise FileNotFoundError(f'Results JSON not found: {results_path}')

with open(results_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

model_cfg = results.get('model', {})
latent_dim = int(model_cfg.get('latent_dim', 0))
if latent_dim <= 0:
    raise ValueError('Invalid latent_dim in results JSON')

hidden_dims = model_cfg.get('hidden_dims', None)
if MODEL_TYPE is None:
    model_type = 'ae' if hidden_dims is not None else 'linear'
else:
    model_type = str(MODEL_TYPE).strip().lower()

if model_type not in {'ae', 'linear'}:
    raise ValueError("MODEL_TYPE must be 'linear' or 'ae'")
if model_type == 'ae' and hidden_dims is None:
    raise ValueError('hidden_dims missing in results JSON for AE model')

input_dim = int(model_cfg.get('input_dim', x_train.shape[1]))
if input_dim != x_train.shape[1]:
    raise ValueError(f'Input dim mismatch: json={input_dim}, data={x_train.shape[1]}')

if model_type == 'linear':
    model = LinearAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
else:
    model = AutoEncoder(input_dim=input_dim, latent_dim=latent_dim, hidden_dims=hidden_dims).to(device)

weights_path = WEIGHTS_OVERRIDE_PATH
if weights_path is None:
    weights_path = results.get('artifacts', {}).get('best_model_path', None)

if weights_path is None:
    run_name = results.get('run', 'run')
    if model_type == 'linear':
        weights_path = f'linear_AE_best_{run_name}.pt'
    else:
        weights_path = f'AE_best_{run_name}.pt'

base_dir = DRIVE_ROOT if IS_COLAB else project_root
weights_path = resolve_path(weights_path, base_dir=base_dir, results_dir=results_path.parent)
if weights_path is None or not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')

checkpoint = torch.load(weights_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

print(f'Model type: {model_type} | latent_dim={latent_dim} | input_dim={input_dim}')
print(f'Loaded weights: {weights_path}')

## Step 5: Latent Space Analysis
Encode the matrices into the latent space and analyze feature distributions.

In [ ]:
def compute_latents(model: nn.Module, x_tensor: torch.Tensor, batch_size: int = 256):
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    latents = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model.encoder(xb)
            latents.append(z.cpu().numpy())

    return np.concatenate(latents, axis=0)


latents_test = compute_latents(model, x_test, batch_size=256)
latent_dim = latents_test.shape[1]
latent_cols = [f'z{i + 1}' for i in range(latent_dim)]
latent_df = pd.DataFrame(latents_test, columns=latent_cols)

print('Latent distribution summary (test set):')
display(latent_df.describe().T)

valid_cols = [col for col in latent_cols if latent_df[col].notna().any() and latent_df[col].nunique() > 1]
latent_df = latent_df[valid_cols]

analysis_dir = results_path.parent / 'analysis_outputs'
analysis_dir.mkdir(parents=True, exist_ok=True)
latent_csv_path = analysis_dir / 'latent_test.csv'
latent_df.to_csv(latent_csv_path, index=False)
print(f'Saved latent samples: {latent_csv_path}')

if len(valid_cols) < 2:
    print('Not enough valid latent dimensions for pairwise plots.')
else:
    grid = sns.PairGrid(latent_df, corner=True, diag_sharey=False)
    grid.map_lower(sns.scatterplot, s=12, alpha=0.6, color='#2a9d8f')
    grid.fig.suptitle('Pairwise latent dimension plots', y=1.02)
    pairplot_path = analysis_dir / 'latent_pairwise.png'
    grid.savefig(pairplot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved latent pairwise plot: {pairplot_path}')

## Step 6: Reconstruction Performance
Reconstruct matrices and compute MSE, MAE, and Frobenius norm on the test set.

In [ ]:
def reconstruct_matrices(model: nn.Module, x_tensor: torch.Tensor, n_assets: int, batch_size: int = 64):
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    outputs = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy()
            outputs.append(pred)

    recon_flat = np.concatenate(outputs, axis=0)
    recon = recon_flat.reshape(-1, n_assets, n_assets).astype(np.float32)
    return recon


def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats


recon_corr_test = reconstruct_matrices(model, x_test, n_assets=n_assets, batch_size=64)
errors_df, errors_stats = reconstruction_errors(test_np, recon_corr_test)

print('Reconstruction error statistics (Test Set):')
display(errors_stats)

analysis_dir = results_path.parent / 'analysis_outputs'
analysis_dir.mkdir(parents=True, exist_ok=True)
errors_csv_path = analysis_dir / 'reconstruction_errors_test.csv'
errors_stats_path = analysis_dir / 'reconstruction_error_stats_test.json'

errors_df.to_csv(errors_csv_path, index=False)
with open(errors_stats_path, 'w', encoding='utf-8') as f:
    json.dump(errors_stats.astype(float).to_dict(), f, indent=4)

print(f'Saved reconstruction errors: {errors_csv_path}')
print(f'Saved reconstruction error stats: {errors_stats_path}')

## Step 7: MST Preservation Analysis
Compare the minimum spanning trees (MST) of original vs reconstructed correlation matrices.

In [ ]:
try:
    import networkx as nx
except ImportError as exc:
    raise ImportError('NetworkX is required for MST analysis. Install with: pip install networkx') from exc

try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError('Plotly is required for interactive MST plots. Install with: pip install plotly') from exc

MST_SAMPLE_IDX = 0  # Test set index to visualize
MST_TOPK_FRACTION = 0.1  # Fraction of top nodes by betweenness
MST_SAVE_PREFIX = 'mst_test_idx'

GICS_CSV_PATH = None
if IS_COLAB:
    if DRIVE_ROOT is None:
        raise ValueError('DRIVE_ROOT is not set for Colab')
    GICS_CSV_PATH = DRIVE_ROOT / 'dataset_tesi' / 'gics_by_ticker.csv'
else:
    project_root_local = project_root if 'project_root' in globals() else Path.cwd()
    candidate_paths = [
        project_root_local / 'dataset_tesi' / 'gics_by_ticker.csv',
        project_root_local / 'results' / 'gics_by_ticker.csv',
        project_root_local.parent / 'dataset_tesi' / 'gics_by_ticker.csv',
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            GICS_CSV_PATH = candidate
            break

SECTOR_ORDER = [
    'Energy',
    'Materials',
    'Industrials',
    'Consumer Discretionary',
    'Consumer Staples',
    'Health Care',
    'Financials',
    'Information Technology',
    'Communication Services',
    'Utilities',
    'Real Estate',
    'Unknown',
]
SECTOR_COLORS = [
    '#1f77b4',  # Energy
    '#ff7f0e',  # Materials
    '#2ca02c',  # Industrials
    '#d62728',  # Consumer Discretionary
    '#9467bd',  # Consumer Staples
    '#8c564b',  # Health Care
    '#e377c2',  # Financials
    '#7f7f7f',  # Information Technology
    '#bcbd22',  # Communication Services
    '#17becf',  # Utilities
    '#aec7e8',  # Real Estate
    '#ffffff',  # Unknown
]
COLOR_MAP = {sector: color for sector, color in zip(SECTOR_ORDER, SECTOR_COLORS)}


def load_gics_map(gics_path: Path):
    if gics_path is None or not Path(gics_path).exists():
        print('GICS mapping not found. Using "Unknown" for all nodes.')
        return {}
    gics_df = pd.read_csv(gics_path)
    gics_df['ticker'] = gics_df['ticker'].astype(str).str.upper()
    gics_df['gics_category'] = gics_df['gics_category'].astype(str)
    print(f'Loaded GICS mapping: {gics_path}')
    return dict(zip(gics_df['ticker'], gics_df['gics_category']))


def build_gics_color_map(labels, gics_map):
    node_sectors = {}
    node_colors = {}
    for idx, label in enumerate(labels):
        ticker = str(label).upper()
        sector = gics_map.get(ticker, 'Unknown')
        node_sectors[idx] = sector
        node_colors[idx] = COLOR_MAP.get(sector, '#9e9e9e')
    return node_sectors, node_colors


def extract_and_sanitize_correlation_matrix(corr_matrix: np.ndarray):
    corr = np.array(corr_matrix, dtype=float)
    corr = np.clip(corr, -1.0, 1.0)
    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 1.0)
    return corr


def build_distance_matrix(corr_matrix: np.ndarray):
    dist = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist = 0.5 * (dist + dist.T)
    np.fill_diagonal(dist, 0.0)
    return dist


def build_mst_from_corr(corr_matrix: np.ndarray):
    corr = extract_and_sanitize_correlation_matrix(corr_matrix)
    dist = build_distance_matrix(corr)
    graph = nx.from_numpy_array(dist)
    mst = nx.minimum_spanning_tree(graph, weight='weight')
    return mst


def get_asset_labels(n_assets_local: int):
    candidates = None
    for meta in (train_meta, val_meta, test_meta):
        if isinstance(meta, dict):
            for key in ('tickers', 'asset_names', 'assets'):
                if key in meta and meta[key] is not None:
                    candidates = list(meta[key])
                    break
        if candidates is not None:
            break
    if candidates is not None and len(candidates) == n_assets_local:
        return [str(x) for x in candidates]
    return [f'A{i}' for i in range(n_assets_local)]


def enforce_min_distance(positions, min_dist=0.085, max_iter=600, step=0.35):
    keys = list(positions.keys())
    arr = np.array([positions[k] for k in keys], dtype=float)

    for _ in range(max_iter):
        moved = False
        for i in range(len(arr)):
            for j in range(i + 1, len(arr)):
                delta = arr[j] - arr[i]
                dist = np.linalg.norm(delta)
                if dist < 1e-9:
                    direction = np.random.randn(2)
                    direction /= np.linalg.norm(direction) + 1e-12
                    arr[j] += 1e-3 * direction
                    moved = True
                    continue
                if dist < min_dist:
                    direction = delta / dist
                    shift = 0.5 * (min_dist - dist) * step
                    arr[i] -= shift * direction
                    arr[j] += shift * direction
                    moved = True
        arr = np.clip(arr, -1.0, 1.0)
        if not moved:
            break

    mins = arr.min(axis=0)
    maxs = arr.max(axis=0)
    span = np.maximum(maxs - mins, 1e-12)
    arr = (arr - mins) / span
    arr = arr * 1.9 - 0.95
    return {k: arr[idx] for idx, k in enumerate(keys)}


def plot_mst_graph(mst, labels, title, save_path, pos=None, node_sectors=None, node_colors=None):
    node_names = list(mst.nodes())
    degrees = {i: d for i, d in mst.degree()}

    if pos is None:
        num_nodes = max(1, len(node_names))
        spacing_k = 2.4 / np.sqrt(num_nodes)
        pos = nx.spring_layout(mst, seed=42, k=spacing_k, iterations=600)
        pos = enforce_min_distance(pos, min_dist=0.09, max_iter=700, step=0.4)

    if node_sectors is None:
        node_sectors = {i: 'Unknown' for i in node_names}
    if node_colors is None:
        node_colors = {i: '#7fb3d5' for i in node_names}

    edge_x = []
    edge_y = []
    for u, v in mst.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode='lines',
        line=dict(color='rgba(120, 120, 120, 0.45)', width=1),
        hoverinfo='skip',
        showlegend=False,
    )

    sector_nodes = {}
    for node in node_names:
        sector = node_sectors.get(node, 'Unknown')
        sector_nodes.setdefault(sector, []).append(node)

    node_traces = []
    for sector in SECTOR_ORDER:
        nodes = sector_nodes.get(sector, [])
        if not nodes:
            continue
        xs = [pos[n][0] for n in nodes]
        ys = [pos[n][1] for n in nodes]
        sizes = [12 + 5 * degrees.get(n, 0) for n in nodes]
        hover_text = [f'{labels[n]}<br>Degree: {degrees.get(n, 0)}' for n in nodes]
        node_traces.append(
            go.Scatter(
                x=xs,
                y=ys,
                mode='markers',
                name=sector,
                marker=dict(
                    size=sizes,
                    color=COLOR_MAP.get(sector, '#9e9e9e'),
                    line=dict(color='black', width=0.6),
                    opacity=0.95,
                ),
                text=hover_text,
                hoverinfo='text',
                showlegend=True,
            )
        )

    fig = go.Figure(data=[edge_trace, *node_traces])
    fig.update_layout(
        title=title,
        showlegend=True,
        legend_title_text='GICS Sector',
        hovermode='closest',
        margin=dict(l=20, r=20, t=50, b=20),
        plot_bgcolor='white',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
    )

    save_path = Path(save_path)
    html_path = save_path.with_suffix('.html')
    fig.write_html(html_path, include_plotlyjs='cdn')

    if save_path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.svg', '.pdf'}:
        try:
            fig.write_image(save_path)
        except Exception as exc:
            print(f'Static image export skipped: {exc}')
            print('Install kaleido to enable static image export: pip install kaleido')

    fig.show()
    return pos


def mst_edge_set(mst):
    return {tuple(sorted(edge)) for edge in mst.edges()}


def degree_histogram(mst, num_nodes: int):
    degrees = np.array([d for _, d in mst.degree()], dtype=int)
    counts = np.zeros(num_nodes, dtype=int)
    uniq, freq = np.unique(degrees, return_counts=True)
    counts[uniq] = freq
    return counts


def compare_mst_metrics(orig_mst, recon_mst, topk_fraction=0.1):
    num_nodes = orig_mst.number_of_nodes()
    edges_orig = mst_edge_set(orig_mst)
    edges_recon = mst_edge_set(recon_mst)
    shared_edges = edges_orig & edges_recon
    edge_overlap = len(shared_edges) / max(1, len(edges_orig))

    deg_hist_orig = degree_histogram(orig_mst, num_nodes)
    deg_hist_recon = degree_histogram(recon_mst, num_nodes)
    deg_hist_orig = deg_hist_orig / max(1, deg_hist_orig.sum())
    deg_hist_recon = deg_hist_recon / max(1, deg_hist_recon.sum())
    degree_l1 = float(np.sum(np.abs(deg_hist_orig - deg_hist_recon)))

    avg_path_orig = nx.average_shortest_path_length(orig_mst, weight='weight')
    avg_path_recon = nx.average_shortest_path_length(recon_mst, weight='weight')
    avg_path_ratio = float(avg_path_recon / avg_path_orig) if avg_path_orig > 0 else np.nan

    bet_orig = nx.betweenness_centrality(orig_mst, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(recon_mst, weight='weight', normalized=True)
    bet_df = pd.DataFrame({
        'orig': pd.Series(bet_orig),
        'recon': pd.Series(bet_recon),
    }).fillna(0.0)
    bet_spearman = float(bet_df['orig'].corr(bet_df['recon'], method='spearman'))
    top_k = max(1, int(np.ceil(num_nodes * topk_fraction)))
    top_orig = set(bet_df['orig'].sort_values(ascending=False).head(top_k).index)
    top_recon = set(bet_df['recon'].sort_values(ascending=False).head(top_k).index)
    top_overlap = len(top_orig & top_recon) / top_k

    return {
        'edge_overlap': float(edge_overlap),
        'degree_l1': degree_l1,
        'avg_path_len_orig': float(avg_path_orig),
        'avg_path_len_recon': float(avg_path_recon),
        'avg_path_ratio': float(avg_path_ratio),
        'betweenness_spearman': bet_spearman,
        'betweenness_topk_overlap': float(top_overlap),
    }


if 'recon_corr_test' not in globals():
    recon_corr_test = reconstruct_matrices(model, x_test, n_assets=n_assets, batch_size=64)

if not (0 <= MST_SAMPLE_IDX < test_np.shape[0]):
    raise ValueError(f'MST_SAMPLE_IDX must be in [0, {test_np.shape[0] - 1}]')

analysis_dir = results_path.parent / 'analysis_outputs'
analysis_dir.mkdir(parents=True, exist_ok=True)
mst_dir = analysis_dir / 'mst'
mst_dir.mkdir(parents=True, exist_ok=True)

labels = get_asset_labels(n_assets)
gics_map = load_gics_map(GICS_CSV_PATH)
node_sectors, node_colors = build_gics_color_map(labels, gics_map)

orig_corr = test_np[MST_SAMPLE_IDX]
recon_corr = recon_corr_test[MST_SAMPLE_IDX]
orig_mst = build_mst_from_corr(orig_corr)
recon_mst = build_mst_from_corr(recon_corr)

pos = plot_mst_graph(
    orig_mst,
    labels,
    title=f'Original MST (test idx {MST_SAMPLE_IDX})',
    save_path=mst_dir / f'{MST_SAVE_PREFIX}{MST_SAMPLE_IDX}_original.png',
    pos=None,
    node_sectors=node_sectors,
    node_colors=node_colors,
)
plot_mst_graph(
    recon_mst,
    labels,
    title=f'Reconstructed MST (test idx {MST_SAMPLE_IDX})',
    save_path=mst_dir / f'{MST_SAVE_PREFIX}{MST_SAMPLE_IDX}_reconstructed.png',
    pos=pos,
    node_sectors=node_sectors,
    node_colors=node_colors,
)

sample_metrics = compare_mst_metrics(orig_mst, recon_mst, topk_fraction=MST_TOPK_FRACTION)
print('Sample MST metrics:')
for key, value in sample_metrics.items():
    print(f'  {key}: {value:.6f}' if isinstance(value, float) else f'  {key}: {value}')

all_metrics = []
for idx in range(test_np.shape[0]):
    omst = build_mst_from_corr(test_np[idx])
    rmst = build_mst_from_corr(recon_corr_test[idx])
    metrics = compare_mst_metrics(omst, rmst, topk_fraction=MST_TOPK_FRACTION)
    metrics['index'] = idx
    all_metrics.append(metrics)

metrics_df = pd.DataFrame(all_metrics)
metrics_csv_path = mst_dir / 'mst_metrics_test.csv'
metrics_df.to_csv(metrics_csv_path, index=False)

metrics_summary = metrics_df.drop(columns=['index']).agg(['mean', 'std', 'min', 'median', 'max']).T
summary_json_path = mst_dir / 'mst_metrics_summary_test.json'
with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_summary.astype(float).to_dict(), f, indent=4)

print(f'Saved MST metrics: {metrics_csv_path}')
print(f'Saved MST metrics summary: {summary_json_path}')

plot_specs = [
    ('edge_overlap', 'Edge overlap (share of edges)', (0, 1)),
    ('degree_l1', 'Degree distribution L1 distance', None),
    ('avg_path_ratio', 'Avg shortest path ratio (recon/orig)', None),
    ('betweenness_topk_overlap', 'Top-k betweenness overlap', (0, 1)),
    ('betweenness_spearman', 'Betweenness Spearman correlation', (-1, 1)),
]

for col, title, y_limits in plot_specs:
    fig, ax = plt.subplots(figsize=(8, 4))
    values = metrics_df[col].replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(values, bins=30, color='steelblue', edgecolor='black', alpha=0.75)
    ax.set_title(f'{title} (Test Set)', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    if y_limits is not None:
        ax.set_xlim(y_limits[0], y_limits[1])
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plot_path = mst_dir / f'mst_{col}_hist.png'
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved MST histogram: {plot_path}')